# 02 · 9B 训练不稳定（HANDOFF §6 P1 只完成一半）

**对应 HANDOFF §6 P1**「9B real 极小 count 冒烟，验 hybrid backward / 显存」。

冒烟本身**成功过**：双节点分卡布局下 `needs_offload=False`，绕开了
`torch_memory_saver` 那个卡了五次的断言，`R2-nonanguard` 单个 run 产出
**500 次真实评测**。hybrid backward 与显存都验证过了。

**没完成的是稳定性**：29 个 9B run 至今零存活。


In [ ]:
%matplotlib inline
import json, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
def load(name): return json.loads((R / name).read_text())

runs = load("run_outcomes.json")
c9 = Counter(r["outcome"] for r in runs if r["size"] == "9B")
c15 = Counter(r["outcome"] for r in runs if r["size"] == "1.5B")
n9, n15 = sum(c9.values()), sum(c15.values())
print(f"9B   n={n9}   {dict(c9)}")
print(f"1.5B n={n15}   {dict(c15)}")

## 图 1 — 两类 OOM 是不同的病，修法相反

In [ ]:
order = ["alive", "gpu_oom", "host_oom", "session_kill", "other"]
lab = {"alive": "still running", "gpu_oom": "GPU out of memory",
       "host_oom": "host RAM OOM (cgroup)", "session_kill": "reaped on session switch",
       "other": "other / never started"}
col = {"alive": "#1a9850", "gpu_oom": "#b2182b", "host_oom": "#d6604d",
       "session_kill": "#f4a582", "other": "#bbbbbb"}
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4))
for ax, (size, cnt, n) in zip(axes, [("9B", c9, n9), ("1.5B", c15, n15)]):
    vals = [cnt.get(k, 0) for k in order]
    b = ax.barh(range(len(order)), vals, color=[col[k] for k in order], height=.62)
    ax.set_yticks(range(len(order))); ax.set_yticklabels([lab[k] for k in order], fontsize=7.5)
    ax.invert_yaxis(); ax.set_xlabel("number of runs")
    ax.set_title(f"{size}   n={n}   GPU {100*cnt.get('gpu_oom',0)/n:.0f}% · "
                 f"host {100*cnt.get('host_oom',0)/n:.0f}%", fontsize=9)
    for r_, v in zip(b, vals):
        if v: ax.text(v+.25, r_.get_y()+r_.get_height()/2, str(v), va="center", fontsize=8)
    ax.set_xlim(0, max(vals)*1.3+1); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()

**读法**：**主机内存 OOM 只出现在 9B，1.5B 一个都没有**——9B 加载
checkpoint 的内存峰值大得多。两类的修法方向相反：

| | 现象 | 能不能避 |
|---|---|---|
| GPU 显存不足 | `torch.OutOfMemoryError` | 基本不能，`nodelock` 已禁用 |
| 主机内存 OOM | `slurmstepd: oom_kill event` | **能**，起跑前查 job cgroup 余量 |

这两类此前都躲在「其他」里：分类器最初只认 `torch.OutOfMemoryError`
（漏了一个类别），补上之后 `R1-base-v10` 仍被误分，因为它的 `oom_kill`
只出现在 `ray_worker.log` 而分类器只读 `train.log`（漏了一个文件）。
补齐两处后「其他」从 10 缩到 3。

## 主机内存 OOM 的根因：限制在 job 级 cgroup

```
job_6247827   memory.max     = 449 GB     ← Slurm 给整个分配的上限
              memory.current = 159 GB
              step 数        = 5          ← 我的 9B + 另一条战线的评测进程
节点           MemTotal       = 898 GB
              MemAvailable   = 695 GB     ← 与 cgroup 限制完全无关
step 级        memory.max     = max        ← step 自己没有上限
```

**同一分配下所有 attach 进来的 step 共享那 449 GB，先到先得。**

第一版预检查 `scontrol show node` 的 `FreeMem`，门槛 200 GB——
**永远不会触发**：实测所有节点 FreeMem 都是 350–770 GB，而出事那台
正是在 FreeMem 673 GB 时被 OOM 掉的。一个永远不触发的检查比没有检查更糟。

现在读 job cgroup 的 `memory.max − memory.current`，门槛 **300 GB**，
从两次失败反推：v9 和 v10 都在余 289 GB 的分配上死，同期余 449 GB 的
分配上 actor 加载正常。**这个检查已经在生产中拦下过一次起跑**
（`R1-base-v11`，目标分配只剩 259 GB）。

## 还要做什么

1. **压低 9B 的内存峰值**：`--rollout-num-gpus 4` 会起 4 个独立 sglang 引擎，
   每个都把完整的 9B 读进主机内存。降到 2 个能减半，代价是一半推理吞吐。
2. **失败要便宜**：起完 3 分钟内回查 step 是否还在，死了立刻换节点重起。
3. **9B 只起在 4/4 真空且 cgroup 余量 ≥300 GB 的节点对上**。